# Plant Disease Detection — Transfer Learning, Grad-CAM and Domain Shift

This is a **thin** notebook: all real code lives in `src/`. The notebook only calls it in order.

**Run order on Colab:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`
(or run the cells top to bottom). Total time ≈ 40 min on a T4 (6 training runs).

| Section | What happens |
|---|---|
| 0 | Settings (put your GitHub URL here) |
| 1 | GPU check, clone repo, install requirements, mount Google Drive |
| 2 | Download + split PlantVillage (lab, with leaf masks) and PlantDoc (field) |
| 3 | Baseline CNN from scratch |
| 4 | Transfer learning: EfficientNet-B0 |
| 5 | Evaluation (accuracy, precision/recall/F1, confusion matrix) |
| 6 | Domain shift: PlantVillage → PlantDoc |
| 7 | Stronger augmentation to reduce the gap (before / after) |
| 8 | Background-bias experiment (original / leaf-only / random background) |
| 9 | Grad-CAM heatmaps + leaf-focus score |
| 10 | Confidence threshold: rejecting unsure predictions |
| 11 | Save everything to Drive, download results |
| 12 | (Optional) Gradio demo |

## 0. Settings

In [ ]:
# The GitHub repository this notebook clones on Colab
GITHUB_REPO_URL = "https://github.com/nandshiv/Plant-disease-Detection.git"

SEED = 42
SMOKE_TEST = False   # True = tiny data + 1 epoch, only to check that everything runs

if SMOKE_TEST:
    TRAIN_PER_CLASS, VAL_PER_CLASS, TEST_PER_CLASS = 2, 1, 1
    PLANTDOC_MAX_PER_CLASS, PLANTDOC_SPLITS = 2, ("test",)
    CNN_EPOCHS = EFFNET_EPOCHS = 1
else:
    # up to 200/40/60 images per class x 38 classes = 7,430 / 1,520 / 2,244 images
    # (a few small classes have fewer; see scripts/dataset_stats.py)
    TRAIN_PER_CLASS, VAL_PER_CLASS, TEST_PER_CLASS = 200, 40, 60
    PLANTDOC_MAX_PER_CLASS, PLANTDOC_SPLITS = None, ("train", "test")  # all ~2,500 field images
    CNN_EPOCHS = 12      # scratch CNN needs more epochs
    EFFNET_EPOCHS = 6    # pretrained model converges fast

CNN_LR, EFFNET_LR = 1e-3, 5e-4
BATCH_SIZE = 64
LEAF_FOCUS_IMAGES = 20 if SMOKE_TEST else 300   # test images used for the leaf-focus score

## 1. Setup: GPU, code, packages, Google Drive
On Colab the repo is cloned to `/content/<repo>` and checkpoints go to
`MyDrive/plant_disease/checkpoints`, so they survive if the runtime disconnects.
Off Colab (e.g. on the laptop) these steps are skipped.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !nvidia-smi
import torch
print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
elif IN_COLAB:
    print("WARNING: no GPU! Runtime -> Change runtime type -> T4 GPU, then restart.")

In [ ]:
if IN_COLAB:
    REPO_DIR = "/content/" + GITHUB_REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.exists(REPO_DIR):
        !git clone {GITHUB_REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull
    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
else:
    # Running locally from notebooks/: go to the project root
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
    REPO_DIR = os.getcwd()
sys.path.insert(0, REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/plant_disease"
    CKPT_DIR = os.path.join(DRIVE_DIR, "checkpoints")   # checkpoints straight to Drive
    DATA_DIR = "/content/data"                          # Colab local disk (fast)
else:
    DRIVE_DIR = None
    CKPT_DIR = "results/checkpoints"
    DATA_DIR = "data"
RESULTS_DIR = "results"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("checkpoints ->", CKPT_DIR, "| results ->", RESULTS_DIR, "| data ->", DATA_DIR)

## 2. Data
* **PlantVillage** (Hugging Face `mohanty/PlantVillage`, no login): 54k lab photos, 38 classes.
  We take a balanced subset. **Test** comes from the authors' official test split; **train/val**
  are split from the official train split **by leaf**, so no physical leaf is in two splits (no leakage).
* **PlantDoc** (GitHub `pratikkayal/PlantDoc-Dataset`): ~2.5k real field photos. Used **only for testing**,
  on the 28 classes that also exist in PlantVillage.

In [ ]:
from src.utils import set_seed, get_device
from src.data import (prepare_plantvillage, prepare_plantdoc, PV_CLASSES, OVERLAP_CLASSES,
                      PLANTDOC_TO_PLANTVILLAGE, LeafDataset, get_transforms)
set_seed(SEED)
device = get_device()
NUM_WORKERS = min(4, os.cpu_count() or 1) if IN_COLAB else 0

PV_DIR = prepare_plantvillage(DATA_DIR, TRAIN_PER_CLASS, VAL_PER_CLASS, TEST_PER_CLASS,
                              seed=SEED, remote=not IN_COLAB)
PLANTDOC_DIR = prepare_plantdoc(DATA_DIR, splits=PLANTDOC_SPLITS,
                                max_per_class=PLANTDOC_MAX_PER_CLASS)
print(len(PV_CLASSES), "PlantVillage classes,", len(OVERLAP_CLASSES), "shared with PlantDoc")
for split in ["train", "val", "test"]:
    print(f"PlantVillage {split}: {len(LeafDataset(os.path.join(PV_DIR, split)))} images")
print("PlantDoc (field):", len(LeafDataset(PLANTDOC_DIR)), "images")

In [ ]:
# Look at the data: top row = PlantVillage (lab), bottom row = PlantDoc (field), same classes
import matplotlib.pyplot as plt
from PIL import Image
show = OVERLAP_CLASSES[::5][:6]
fig, axes = plt.subplots(2, len(show), figsize=(3 * len(show), 6))
for j, cls in enumerate(show):
    for i, root in enumerate([os.path.join(PV_DIR, "train"), PLANTDOC_DIR]):
        folder = os.path.join(root, cls)
        files = sorted(os.listdir(folder)) if os.path.isdir(folder) else []
        if files:
            axes[i, j].imshow(Image.open(os.path.join(folder, files[0])))
        axes[i, j].set_title(("lab: " if i == 0 else "field: ") + cls.split("___")[0][:12], fontsize=9)
        axes[i, j].axis("off")
plt.tight_layout(); plt.show()

## 3. Baseline: small CNN trained from scratch
4 conv blocks + global average pooling (~1.2M parameters), random initial weights.
Training augmentation is "standard" (only flips). The checkpoint with the best **validation** accuracy is kept.

In [ ]:
from src.train import run_experiment
from IPython.display import Image as ShowImage, display

ckpts = {}
ckpts["cnn_standard"] = run_experiment("cnn_standard", "cnn", "standard", PV_DIR, CNN_EPOCHS, CNN_LR,
                                       RESULTS_DIR, CKPT_DIR, BATCH_SIZE, NUM_WORKERS, SEED)
display(ShowImage(f"{RESULTS_DIR}/plots/history_cnn_standard.png"))

## 4. Transfer learning: EfficientNet-B0 pretrained on ImageNet
Same data, same augmentation. Only the last layer is new (38 outputs); all other layers start from
ImageNet weights and are **fine-tuned** with a small learning rate.

In [ ]:
ckpts["effnet_standard"] = run_experiment("effnet_standard", "effnet", "standard", PV_DIR, EFFNET_EPOCHS,
                                          EFFNET_LR, RESULTS_DIR, CKPT_DIR, BATCH_SIZE, NUM_WORKERS, SEED)
display(ShowImage(f"{RESULTS_DIR}/plots/history_effnet_standard.png"))

## 5. Evaluation on the PlantVillage test set
For each model: accuracy, per-class precision / recall / F1 (saved as JSON) and a row-normalised
confusion matrix (diagonal = recall of each class). `evaluate_experiment` also runs the PlantDoc test (next section).

In [ ]:
from src.models import load_checkpoint
from src.evaluate import evaluate_experiment, results_table, plot_domain_shift
from src.utils import save_json

summaries = {}
for name in ["cnn_standard", "effnet_standard"]:
    model, _, _ = load_checkpoint(ckpts[name], device)
    summaries[name] = evaluate_experiment(model, name, PV_DIR, PLANTDOC_DIR, device,
                                          RESULTS_DIR, NUM_WORKERS)
display(ShowImage(f"{RESULTS_DIR}/eval/effnet_standard_pv_test_confusion.png", width=700))

In [ ]:
# Per-class report for the transfer-learning model (lowest F1 first)
from src.utils import load_json
per_class = load_json(f"{RESULTS_DIR}/eval/effnet_standard_pv_test_metrics.json")["per_class"]
for cls, m in sorted(per_class.items(), key=lambda kv: kv[1]["f1-score"])[:10]:
    print(f"{cls:55s} P={m['precision']:.2f} R={m['recall']:.2f} F1={m['f1-score']:.2f} n={m['support']:.0f}")

## 6. Domain shift: lab photos → field photos
We compare accuracy on PlantVillage test vs. PlantDoc **on the same 28 shared classes**.
The model still chooses among all 38 classes (it doesn't know PlantDoc has fewer), like a real app would.
**Domain gap = PlantVillage accuracy − PlantDoc accuracy.**

In [ ]:
print(results_table(summaries))
display(ShowImage(f"{RESULTS_DIR}/eval/effnet_standard_plantdoc_confusion.png", width=700))

## 7. Reducing the gap with stronger augmentation
Field photos have different zoom, angles, lighting and focus. The **strong** training augmentation
imitates that: `RandomResizedCrop(scale 0.5-1)`, flips, `RandomRotation(30°)`, `ColorJitter`, `GaussianBlur`.
We retrain both models with it (everything else identical) and compare **before / after**.

In [ ]:
ckpts["cnn_strong"] = run_experiment("cnn_strong", "cnn", "strong", PV_DIR, CNN_EPOCHS, CNN_LR,
                                     RESULTS_DIR, CKPT_DIR, BATCH_SIZE, NUM_WORKERS, SEED)
ckpts["effnet_strong"] = run_experiment("effnet_strong", "effnet", "strong", PV_DIR, EFFNET_EPOCHS,
                                        EFFNET_LR, RESULTS_DIR, CKPT_DIR, BATCH_SIZE, NUM_WORKERS, SEED)
for name in ["cnn_strong", "effnet_strong"]:
    model, _, _ = load_checkpoint(ckpts[name], device)
    summaries[name] = evaluate_experiment(model, name, PV_DIR, PLANTDOC_DIR, device,
                                          RESULTS_DIR, NUM_WORKERS)

In [ ]:
# Final comparison: this table goes into the README
table = results_table(summaries)
print(table)
with open(f"{RESULTS_DIR}/results_table.md", "w") as f:
    f.write(table + "\n")
save_json(summaries, f"{RESULTS_DIR}/summary_all.json")
plot_domain_shift(summaries, f"{RESULTS_DIR}/plots/domain_shift.png")
display(ShowImage(f"{RESULTS_DIR}/plots/domain_shift.png"))

## 8. Background-bias experiment: does the model look at the leaf or at the background?
PlantVillage photos all have a similar plain background, so a model could partly learn the *background*
(a "shortcut"). We retrain the baseline CNN with two changed training sets, using the leaf masks:

| Variant | Training images |
|---|---|
| `cnn_standard` (already trained) | original photos |
| `cnn_leaf_only` | background set to black, only the leaf left |
| `cnn_random_bg` | leaf pasted on a new random blotchy background every time it is loaded |

Then every model is tested on the **same** test leaves with all three backgrounds, and on PlantDoc.
A big drop when only the background changes = the model relied on the background.

In [ ]:
from src.evaluate import evaluate_background_bias, background_table, plot_background_bias

for bg in ["leaf_only", "random_bg"]:
    name = f"cnn_{bg}"
    ckpts[name] = run_experiment(name, "cnn", "standard", PV_DIR, CNN_EPOCHS, CNN_LR, RESULTS_DIR,
                                 CKPT_DIR, BATCH_SIZE, NUM_WORKERS, SEED, background=bg)
    model, _, _ = load_checkpoint(ckpts[name], device)
    summaries[name] = evaluate_experiment(model, name, PV_DIR, PLANTDOC_DIR, device,
                                          RESULTS_DIR, NUM_WORKERS)

bg_results = {}
for name in ["cnn_standard", "cnn_leaf_only", "cnn_random_bg", "effnet_standard", "effnet_strong"]:
    model, _, _ = load_checkpoint(ckpts[name], device)
    bg_results[name] = evaluate_background_bias(model, name, PV_DIR, PLANTDOC_DIR, device,
                                                RESULTS_DIR, NUM_WORKERS, SEED)
table = background_table(bg_results)
print(table)
with open(f"{RESULTS_DIR}/background_table.md", "w") as f:
    f.write(table + "\n")
plot_background_bias(bg_results, f"{RESULTS_DIR}/plots/background_bias.png")
display(ShowImage(f"{RESULTS_DIR}/plots/background_bias.png"))

In [ ]:
# Updated main table (now also with the two background variants)
table = results_table(summaries)
print(table)
with open(f"{RESULTS_DIR}/results_table.md", "w") as f:
    f.write(table + "\n")
save_json(summaries, f"{RESULTS_DIR}/summary_all.json")

## 9. Grad-CAM: where does the model look? + leaf-focus score
Red = regions that increased the score of the predicted class. We show correct **and** wrong predictions,
on lab images and on field images. A good model looks at the lesions on the leaf, not at the background.

**Leaf-focus score** = share of the Grad-CAM heat that falls inside the leaf mask
(`leaf_focus_score` in `src/gradcam.py`). Compare it with the leaf's share of the image area
(`mean_leaf_area`): an evenly spread heatmap would score exactly that. Only possible on PlantVillage
(PlantDoc has no masks).

In [ ]:
from src.gradcam import save_gradcam_examples, evaluate_leaf_focus
from src.models import get_gradcam_layer
from src.data import pv_dataset

datasets_for_cam = {"pv_test": pv_dataset(PV_DIR, "test"),   # has masks -> leaf focus in titles
                    "plantdoc": LeafDataset(PLANTDOC_DIR, get_transforms("eval"))}
for name in ["cnn_standard", "cnn_random_bg", "effnet_standard", "effnet_strong"]:
    model, model_name, class_names = load_checkpoint(ckpts[name], device)
    for ds_name, ds in datasets_for_cam.items():
        out = f"{RESULTS_DIR}/gradcam/{name}_{ds_name}.png"
        save_gradcam_examples(model, get_gradcam_layer(model, model_name), ds, class_names,
                              device, out, f"{name} on {ds_name}", n_correct=4, n_incorrect=4,
                              seed=SEED)
display(ShowImage(f"{RESULTS_DIR}/gradcam/effnet_strong_plantdoc.png", width=500))

In [ ]:
focus = {}
for name in ckpts:
    model, model_name, _ = load_checkpoint(ckpts[name], device)
    focus[name] = evaluate_leaf_focus(model, get_gradcam_layer(model, model_name),
                                      pv_dataset(PV_DIR, "test"), device,
                                      f"{RESULTS_DIR}/leaf_focus/{name}.json",
                                      max_images=LEAF_FOCUS_IMAGES, seed=SEED)
lines = ["| Model | Leaf focus (mean) | Leaf focus (median) | Leaf area (mean) | Focus when correct | Focus when wrong |",
         "|---|---|---|---|---|---|"]
fmt = lambda v: "n/a" if v is None else f"{v:.3f}"
for name, r in focus.items():
    lines.append(f"| {name} | {fmt(r['mean_leaf_focus'])} | {fmt(r['median_leaf_focus'])} | "
                 f"{fmt(r['mean_leaf_area'])} | {fmt(r['mean_leaf_focus_correct'])} | {fmt(r['mean_leaf_focus_wrong'])} |")
print("\n".join(lines))
with open(f"{RESULTS_DIR}/leaf_focus_table.md", "w") as f:
    f.write("\n".join(lines) + "\n")

## 10. Confidence threshold: say "not sure" instead of guessing
The model always outputs *some* class, even for a strange photo. We reject a prediction when its
confidence (softmax probability of the predicted class) is below a threshold.

**How the threshold is chosen (validation set only):** the confidence below which only 5% of the
*correct* validation predictions fall (`keep_correct=0.95` in `rejection_analysis`). Then we report on the
test sets: **coverage** (share of images answered) and **accuracy on the answered images**.
Trade-off: a higher threshold → fewer answers, but more of them are right.

In [ ]:
from src.evaluate import rejection_analysis
rejection = {}
for name in ["effnet_standard", "effnet_strong"]:
    model, _, _ = load_checkpoint(ckpts[name], device)
    rejection[name] = rejection_analysis(model, name, PV_DIR, PLANTDOC_DIR, device, RESULTS_DIR,
                                         keep_correct=0.95, num_workers=NUM_WORKERS)
display(ShowImage(f"{RESULTS_DIR}/rejection/effnet_strong_coverage_accuracy.png"))

## 11. Save results to Google Drive and download them
Checkpoints are already on Drive (`MyDrive/plant_disease/checkpoints`). This copies the `results/`
folder (metrics JSON, plots, confusion matrices, Grad-CAM, leaf focus, background bias, rejection) to
`MyDrive/plant_disease/results` and also downloads a zip to your computer.

In [ ]:
import shutil
shutil.make_archive("results_export", "zip", RESULTS_DIR)
if IN_COLAB:
    shutil.copytree(RESULTS_DIR, os.path.join(DRIVE_DIR, "results"), dirs_exist_ok=True)
    shutil.copy("results_export.zip", DRIVE_DIR)
    print("Copied to", DRIVE_DIR)
    from google.colab import files
    files.download("results_export.zip")
else:
    print("Saved results_export.zip in", os.getcwd())

## 12. (Optional) Gradio demo inside Colab
Starts the app with a public `*.gradio.live` link (valid ~72 h). It uses the rejection threshold saved in
section 10. The cell keeps running while the app is up; press the stop button to end it.

In [ ]:
RUN_DEMO = False   # set True and run this cell to start the demo
if RUN_DEMO:
    !python app.py --share --ckpt "{CKPT_DIR}/effnet_strong.pt"